# 04. e + Ba: сечение и константы скоростей для 553.548 нм

* A и ветвление — NIST ASD: 6s6p ¹P₁ → 6s² (553.548, 1.19·10⁸, A+), → 6s5d ¹D₂ (1499.985,
  2.5·10⁵, B), → ³D₂ (1130.303, 1.1·10⁵, C), → ³D₁ (1107.570, 3.1·10³, D+): BR = 0.997.
* σ из основного состояния — **готово**: Fursa et al, PRA 60, 4590 (1999), табл. II (прямое
  Q и «apparent» Q_app = Chen & Gallagher 1976 × 1.03), 2.5–897.6 эВ; ниже 5 эВ только
  точки 2.5, 3, 4 эВ.
* σ из метастабилей 6s5d ³D₁,₂,₃, ¹D₂ в ¹P₁ и подробная околопороговая область —
  **считаются здесь**: модель `ba1` (`models.py`): два валентных электрона над [Xe],
  состояния до 4.1 эВ.

Проверки: f(553.5) и A против NIST; σ(¹P₁) против Fursa (прямое) и Q_app (Chen & Gallagher)
с каскадами из этого же расчёта; остальные σ против Fursa, табл. III–VI.

Время: мишень и A — 2 мин 20 с (замер, 2 ядра; 39 из 49 состояний сопоставлены с NIST); рассеяние
не замерено, оценка — сутки на сервере.

**Выход** (папка `ba1`): `sigma_ba1.xlsx`, `rates_ba1.xlsx` (все возбуждения, формат Ванга),
`ba1_line_sigma.xlsx`, `ba1_k_line.xlsx` (+ .csv; для 6s² — и по Fursa, и по DBSR).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
ROOT    = Path("/data/files_Otdel120/pydbsr_calc")    # папка установки (большой диск)
REPO    = ROOT / "pydbsr_src"                         # распакованный архив pydbsr (или клон репозитория)
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = ROOT                                        # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = ROOT / "runs" / "crm"                       # результаты расчётов (большой диск)
XE2_RUN = ROOT / "runs" / "xe_plus_v2"                # готовый расчёт e + Xe+ (модель Ванга, J <= 25)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

## 0. Готовое: σ линии 553.5 из основного состояния по Fursa et al (1999), табл. II

In [ ]:
x553 = nb.BA_LINES[0]
F = nb.FURSA_1P1
TEa = np.array(TE)
k_fursa = x553["branching"] * rates.rates_on_grid(F["E"], F["Q"] * 1e-16, TEa, 18060.261 / nb.CM_PER_EV, ion=False)
print("σ ниже 2.5 эВ: линейно от порога 2.239 эВ до первой точки (ion=False)")
for t, k in zip(TEa, k_fursa):
    print(f"Te = {t:5.2f} эВ   BR·<σv> = {k:.3e} см3/с")

In [ ]:
JMAX = 0            # 0 (проверка), затем 15-25
job = nb.Job(RUNS / "ba1", "ba1", stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=40, de=0.005, te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

In [ ]:
W = RUNS / "ba1"
print((W / "target_table.txt").read_text())

## 1. A и gf против NIST

In [ ]:
nistA = [x for x in nb.nist_lines(NIST / "nist_BaI.csv") + nb.nist_lines(NIST / "nist_BaI_IR.csv") if x["A"]]
for r in nb.read_csv(W / "transitions_E1.csv"):
    wl = nb.fnum(r["wl_air_nm"])
    hit = [x for x in nistA if wl and abs(x["wl"] - wl) < 0.02]
    if hit:
        x = hit[0]; a = float(r["A_exp"])
        print(f"{x['wl']:9.3f} {r['upper_nist']:>18s} -> {r['lower_nist']:<16s} DBSR {a:.3e}  NIST {x['A']:.3e}  "
              f"DBSR/NIST {a / x['A']:.2f}  gf {float(r['gf']):.3f}  vel/len {r['gauge_ratio']}")

## 2. σ(6s² → 6s6p ¹P₁): прямое против Fursa, с каскадами против Chen & Gallagher

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz", partial_waves=False)
tg = db.Target.load(W / "target")
st = {s.name: s for s in tg.states if s.name in cs.names}
lab = {n: s.nist_label for n, s in st.items()}
gs = min(st, key=lambda n: st[n].exp_energy_cm)
p1 = [n for n in st if lab[n] == "6s.6p 1P* J=1"][0]
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
num = {n: st[n].nist_no for n in st}
def reach(u, target, depth=4):          # вероятность попасть из u в target через цепочку распадов
    if u == target: return 1.0
    if depth == 0 or u not in br: return 0.0
    return sum(b * reach(l, target, depth - 1) for l, b in br[u].items())
E = cs.incident_energy(gs)
direct = np.nan_to_num(cs.sigma(gs, p1))
casc = sum(np.nan_to_num(cs.sigma(gs, u)) * reach(num[u], num[p1]) for u in st
           if st[u].exp_energy_cm > st[p1].exp_energy_cm)
plt.plot(E, direct / 1e-16, label="DBSR прямое")
plt.plot(E, (direct + casc) / 1e-16, label="DBSR + каскады")
plt.plot(F["E"], F["Q"], "s", ms=4, label="Fursa 1999, табл. II, Q")
plt.plot(F["E"], F["Q_app"], "o", ms=4, label="Fursa 1999, табл. II, Q_app (Chen & Gallagher ×1.03)")
plt.xscale("log"); plt.xlim(2, 1000); plt.xlabel("E, эВ"); plt.ylabel("σ, 10⁻¹⁶ см²"); plt.legend(fontsize=7); plt.show()

## 3. Остальные σ из основного против Fursa (табл. III, IV, VI)

In [ ]:
import re
def find_state(key):                       # "6s6p 3P1" -> состояние с меткой NIST "6s.6p 3P* J=1"
    conf, tj = key.split()
    c = ".".join(re.findall(r"\d[a-z](?:\d(?![a-z]))?", conf))
    want = {f"{c} {tj[:2]} J={tj[2:]}", f"{c} {tj[:2]}* J={tj[2:]}"}
    return [n for n in st if lab[n] in want]
fig, axs = plt.subplots(3, 4, figsize=(15, 9))
for ax, key in zip(axs.flat, nb.FURSA_TABLES):
    Ef, sf = nb.fursa(key)
    ax.plot(Ef, sf / 1e-16, "s", ms=3, label="Fursa")
    for n in find_state(key)[:1]:
        ax.plot(cs.incident_energy(gs), cs.sigma(gs, n) / 1e-16, lw=0.8, label="DBSR")
    ax.set_xscale("log"); ax.set_title(key, fontsize=9)
axs.flat[0].legend(fontsize=7); fig.tight_layout(); plt.show()

## 4. Сечение линии 553.5 из 6s² и из 6s5d; константы скоростей

In [ ]:
sig, labs, k_line = {}, {}, {}
for i in [gs] + [n for n in st if (lab[n] or "").startswith("6s.5d")]:
    Ei, s = cs.incident_energy(i), cs.sigma(i, p1)
    m = np.isfinite(s) & (Ei > 0)
    key = (num[i], num[p1], 553.548)
    sig[key] = (Ei[m], x553["branching"] * s[m])
    labs[key] = f"{num[i]}->{num[p1]} (553.548)"
    k_line[key] = x553["branching"] * cs.rate(i, p1, TEa * 11604.518)
    plt.plot(Ei[m], x553["branching"] * s[m] / 1e-16, lw=0.8, label=f"из {lab[i]}")
plt.xscale("log"); plt.xlabel("E, эВ"); plt.ylabel("σ линии 553.5, 10⁻¹⁶ см²"); plt.legend(fontsize=7); plt.show()
k_line[("Fursa", 9, 553.548)] = k_fursa
labs[("Fursa", 9, 553.548)] = "1->9 (553.548) Fursa1999 tab.II"
lv = [l for l in db.nist.fetch_levels("Ba I") if l.no in set(num.values())]
reference.write_xlsx(W / "ba1_line_sigma.xlsx", lv, {k: v for k, v in sig.items()}, sheet_of=lambda k: "553.548",
                     labels=labs)
reference.write_rates_xlsx(W / "ba1_k_line.xlsx", TEa, k_line, labs,
                           note="BR * <sigma v> (cm3/s); sigma: pydbsr model ba1 (this run) and Fursa et al 1999 tab. II; BR = 0.997 (NIST)")
print("->", W / "ba1_line_sigma.xlsx", W / "ba1_k_line.xlsx")